# SQL Server → raw volume

Reads the CRM customer table over JDBC and lands it as CSV in
`raw_data/<source>/load_date=<run_date>`. No business logic here — the job of
ingestion is to get the bytes in, unchanged, in a folder named by the day.

Re-running a day replaces that day's folder, so retries are safe.

In [ ]:
import sys
from datetime import date
from pathlib import Path

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "common_utils").is_dir():
        sys.path.insert(0, str(candidate))
        break

from common_utils.ingestors import jdbc_url, read_jdbc, write_files
from common_utils.logger import get_logger, log_info
from common_utils.observability import ensure_ops_schema, new_run_id, track
from common_utils.settings import get_secret, load_json, parse_run_date
from common_utils.writers import create_namespace

In [ ]:
dbutils.widgets.text("config_path", "src/ingestion/config/sqlserver_customers.json")
dbutils.widgets.text("catalog", "retaildataplatform")
dbutils.widgets.text("bronze_schema", "bronze")
dbutils.widgets.text("raw_volume", "raw_data")
dbutils.widgets.text("secret_scope", "retail-platform-dev")
dbutils.widgets.text("run_date", date.today().isoformat())

config = load_json(dbutils.widgets.get("config_path"))
catalog = dbutils.widgets.get("catalog")
bronze_schema = dbutils.widgets.get("bronze_schema")
raw_volume = dbutils.widgets.get("raw_volume")
scope = dbutils.widgets.get("secret_scope")
run_date = parse_run_date(dbutils.widgets.get("run_date"))
run_id = new_run_id()

connection = config["connection"]
secrets = config["secret_keys"]
target = f"/Volumes/{catalog}/{bronze_schema}/{raw_volume}/{config['source_name']}/load_date={run_date}"
logger = get_logger("ingestion")

In [ ]:
create_namespace(spark, catalog, bronze_schema, raw_volume, comment="Bronze: raw landing volume and raw copies of source data")
ensure_ops_schema(spark, catalog)

with track(spark, catalog, run_id, run_date, task="sqlserver_ingestion", layer="raw", entity=config["source_name"]) as stats:
    url = jdbc_url(
        host=connection["host"],
        database=connection["database"],
        port=connection.get("port", 1433),
        **connection.get("jdbc_options", {}),
    )
    log_info(logger, "connecting", host=connection["host"], database=connection["database"], table=connection["table"])

    df = read_jdbc(
        spark,
        url=url,
        user=get_secret(dbutils, scope, secrets["username"]),
        password=get_secret(dbutils, scope, secrets["password"]),
        table=connection.get("table"),
        query=connection.get("query"),
        options=connection.get("read_options"),
    )

    rows = df.count()
    write_files(df, target, config["landing_format"])
    stats.rows_read = stats.rows_written = rows
    log_info(logger, "landed", source=config["source_name"], rows=rows, path=target)

In [ ]:
display(spark.read.option("header", "true").csv(target).limit(10))